# Домашнее задание 2 — Planning Agent с ревизией + HITL паттерн

## Описание

Реализуйте двух агентов:
1. **Planning Agent с ревизией** — агент, который строит план, выполняет его и пересматривает при необходимости
2. **HITL-агент с опасными операциями** — Selective Human-in-the-Loop

---

## 📊 Оценка ДЗ (5-балльная шкала)

| Критерий | Баллы |
|---|---|
| Реализован **Planning Agent с ревизией** + **HITL паттерн** | **3** |
| Агент корректно отрабатывает **все тест-кейсы** | **2** |
| **Итого максимум** | **5** |

---
## 0. Подготовка

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import settings
from openai import OpenAI

In [ ]:
client = OpenAI(
    base_url="https://api.polza.ai/api/v1",
    api_key=settings.polza_ai_api_key,
)
MODEL = "deepseek/deepseek-chat-v3.1"

---
## 0.1 База данных городов и инструменты

Код из практики — необходим для выполнения заданий.

In [ ]:
CITY_DATABASE = {
    "Москва": {
        "population": 12_600_000,
        "area_km2": 2561,
        "country": "Россия",
        "founded": 1147,
    },
    "Санкт-Петербург": {
        "population": 5_600_000,
        "area_km2": 1439,
        "country": "Россия",
        "founded": 1703,
    },
    "Новосибирск": {
        "population": 1_630_000,
        "area_km2": 505,
        "country": "Россия",
        "founded": 1893,
    },
    "Екатеринбург": {
        "population": 1_540_000,
        "area_km2": 468,
        "country": "Россия",
        "founded": 1723,
    },
    "Казань": {
        "population": 1_310_000,
        "area_km2": 614,
        "country": "Россия",
        "founded": 1005,
    },
    "Токио": {
        "population": 13_960_000,
        "area_km2": 2194,
        "country": "Япония",
        "founded": 1457,
    },
    "Пекин": {
        "population": 21_540_000,
        "area_km2": 16411,
        "country": "Китай",
        "founded": 1045,
    },
    "Нью-Йорк": {
        "population": 8_336_000,
        "area_km2": 783,
        "country": "США",
        "founded": 1624,
    },
    "Лондон": {
        "population": 8_982_000,
        "area_km2": 1572,
        "country": "Великобритания",
        "founded": 43,
    },
    "Париж": {
        "population": 2_161_000,
        "area_km2": 105,
        "country": "Франция",
        "founded": -250,
    },
}

---
## Задание: HITL-агент с опасными операциями

Реализуйте агента с **Human-in-the-Loop** для базы данных городов, в которой есть **опасные** операции.

### Шаг 1: Добавьте «опасные» инструменты

| Инструмент | Что делает | Опасный? |
|-----------|-----------|---------|
| `search_city` | Читает данные | Нет |
| `list_cities` | Читает список | Нет |
| `calculate` | Вычисляет | Нет |
| `add_city(name, population, area_km2, country, founded)` | Добавляет город в базу | **Да** |
| `update_city(city_name, field, value)` | Изменяет поле города | **Да** |
| `delete_city(city_name)` | Удаляет город из базы | **Да** |

### Шаг 2: Реализуйте Selective HITL

Напишите `run_hitl_agent(question, dangerous_tools, auto_approve=False)`:
- **Безопасные** инструменты (`search_city`, `list_cities`, `calculate`) выполняются автоматически
- **Опасные** инструменты (`add_city`, `update_city`, `delete_city`) требуют подтверждения от человека
- Если человек отклонил — агент получает `{"error": "Действие отклонено пользователем"}` и адаптируется

### Шаг 3: Проверьте на сценарии

Протестируйте агента на запросе:

> *«Добавь в базу город Владивосток (население 600000, площадь 331 км², Россия, основан 1860). Затем найди город России с наименьшим населением.»*

Ожидаемое поведение:
1. Агент вызывает `add_city` → человек одобряет (или отклоняет!)
2. Агент вызывает `list_cities(country="Россия")` → автоматически
3. Агент ищет данные по городам → автоматически
4. Агент даёт ответ

In [2]:
# ============================================================
#  Шаг 1: Реализуйте «опасные» инструменты
# ============================================================
# Подсказка: CITY_DATABASE — это обычный dict.
# Мутировать его из функции можно напрямую


def add_city(name: str, population: int, area_km2: float, country: str, founded: int) -> dict:
    """Добавить новый город в базу данных."""
    # TODO: проверить что города нет в CITY_DATABASE
    # Если есть — вернуть {"error": "Город уже существует"}
    # Если нет — добавить: CITY_DATABASE[name] = {"population": ..., "area_km2": ..., ...}
    # Вернуть {"status": "added", "city": name}
    ...


def update_city(city_name: str, field: str, value) -> dict:
    """Обновить поле существующего города."""
    # TODO: проверить что город есть в CITY_DATABASE
    # Проверить что поле валидное (population, area_km2, country, founded)
    # Обновить: CITY_DATABASE[city_name][field] = value
    # Вернуть {"status": "updated", "city": city_name, "field": field, "new_value": value}
    ...


def delete_city(city_name: str) -> dict:
    """Удалить город из базы данных."""
    # TODO: проверить что город есть в CITY_DATABASE
    # Удалить: del CITY_DATABASE[city_name]
    # Вернуть {"status": "deleted", "city": city_name}
    ...


# TODO: добавьте описания инструментов
# TODO: добавьте функции в маппинг

DANGEROUS_TOOLS_SCHEMA = [
    # TODO: опишите add_city, update_city, delete_city
]

DANGEROUS_TOOL_FUNCTIONS = {
    # TODO: маппинг имя → функция для опасных инструментов
}

In [ ]:
# ============================================================
# Шаг 2: Реализуйте Selective HITL-агент
# ============================================================

# Множество инструментов, требующих одобрения человека
TOOLS_REQUIRING_APPROVAL = {"add_city", "update_city", "delete_city"}


def run_hitl_agent(
    question: str,
    tools_schema: list | None = None,
    tool_functions: dict | None = None,
    dangerous_tools: set | None = None,
    max_steps: int = 10,
    auto_approve: bool = False,
) -> dict:
    """
    Агент с Human-in-the-Loop (selective approval).

    - Безопасные инструменты выполняются автоматически
    - Опасные инструменты требуют подтверждения от человека
    - При отклонении — агент получает ошибку и адаптируется
    """
    # TODO: реализуйте агентный цикл
    # Подсказка: если func_name in dangerous_tools → спрашивайте input("Разрешить? (y/n): ")
    # Если не одобрено — result = {"error": "Действие отклонено пользователем"}
    ...

In [3]:
# ============================================================
# Тест-кейсы для проверки работы агента
# ============================================================

# Кейс 1: Только безопасные операции — поиск информации
# hitl_result = run_hitl_agent(
#     "Добавь в базу город Владивосток (население 600000, площадь 331 км², Россия, основан 1860). "
#     "Затем найди город России с наименьшим населением.",
#     tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
#     tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
#     dangerous_tools=TOOLS_REQUIRING_APPROVAL,
# )

# Кейс 2: Удаление города (опасная операция)
# hitl_result = run_hitl_agent(
#     "Удали Париж из базы данных.",
#     tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
#     tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
#     dangerous_tools=TOOLS_REQUIRING_APPROVAL,
# )

# Кейс 3: Обновление поля города (опасная операция)
# hitl_result = run_hitl_agent(
#     "Обнови население Москвы на 13000000.",
#     tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
#     tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
#     dangerous_tools=TOOLS_REQUIRING_APPROVAL,
# )

# Кейс 4: Безопасный расчёт — плотность населения
# hitl_result = run_hitl_agent(
#     "Вычисли плотность населения (чел/км²) для Пекина.",
#     tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
#     tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
#     dangerous_tools=TOOLS_REQUIRING_APPROVAL,
# )

# Кейс 5: Добавление + удаление (две опасные операции подряд)
# hitl_result = run_hitl_agent(
#     "Добавь город Сочи (население 440000, площадь 176 км², Россия, основан 1838), "
#     "а затем удали Новосибирск из базы.",
#     tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
#     tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
#     dangerous_tools=TOOLS_REQUIRING_APPROVAL,
# )

# Кейс 6: Список всех городов (только безопасное)
# hitl_result = run_hitl_agent(
#     "Перечисли все города Россиии из базы данных.",
#     tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
#     tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
#     dangerous_tools=TOOLS_REQUIRING_APPROVAL,
# )

# Кейс 7: Добавление уже существующего города (ожидаем ошибку от инструмента)
# hitl_result = run_hitl_agent(
#     "Добавь город Казань (население 1310000, площадь 614 км², Россия, основан 1005).",
#     tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
#     tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
#     dangerous_tools=TOOLS_REQUIRING_APPROVAL,
# )

# Кейс 8: Сравнение двух городов (безопасные вызовы + вычисление)
# hitl_result = run_hitl_agent(
#     "Сравни Лондон и Нью-Йорк: у какого города больше плотность населения?",
#     tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
#     tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
#     dangerous_tools=TOOLS_REQUIRING_APPROVAL,
# )

# Кейс 9: Обновление нескольких полей (несколько опасных вызовов)
# hitl_result = run_hitl_agent(
#     "Обнови данные Екатеринбурга: население — 1600000, площадь — 490 км².",
#     tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
#     tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
#     dangerous_tools=TOOLS_REQUIRING_APPROVAL,
# )

# Кейс 10: Смешанный сценарий — безопасный поиск + опасное добавление + расчёт
# hitl_result = run_hitl_agent(
#     "Найди самый старый город в базе. Затем добавь город Дубай "
#     "(население 3500000, площадь 1588 км², ОАЭ, основан 1833) "
#     "и вычисли, во сколько раз население Дубая больше населения Казани.",
#     tools_schema=TOOLS_SCHEMA + DANGEROUS_TOOLS_SCHEMA,
#     tool_functions={**TOOL_FUNCTIONS, **DANGEROUS_TOOL_FUNCTIONS},
#     dangerous_tools=TOOLS_REQUIRING_APPROVAL,
# )